In [4]:
import requests
from bs4 import BeautifulSoup
import string
import time
import json
import os

# --- Configuration ---
LINKS_FILE = "ewg_product_links.txt"
PROGRESS_FILE = "scrape_progress.json"
BASE_URL = "https://www.ewg.org/skindeep/search/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def load_progress():
    """Loads the last saved letter and page number."""
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, 'r') as f:
            return json.load(f)
    return {"letter_idx": 0, "page": 1}

def save_progress(letter_idx, page):
    """Saves current state to prevent data loss."""
    with open(PROGRESS_FILE, 'w') as f:
        json.dump({"letter_idx": letter_idx, "page": page}, f)

def append_links(links):
    """Appends new links to the text file immediately."""
    with open(LINKS_FILE, "a") as f:
        for link in links:
            f.write(f"{link}\n")

def scrape_ewg_with_resume():
    progress = load_progress()
    alphabet = string.ascii_lowercase
    
    start_letter_idx = progress["letter_idx"]
    start_page = progress["page"]

    print(f"Resuming from Letter: {alphabet[start_letter_idx]}, Page: {start_page}")

    for l_idx in range(start_letter_idx, len(alphabet)):
        letter = alphabet[l_idx]
        
        # If we just started a new letter, reset page to 1
        current_page = start_page if l_idx == start_letter_idx else 1
        
        while True:
            print(f"Scraping Letter '{letter}', Page {current_page}...")
            
            params = {'page': current_page, 'search': letter}
            
            try:
                response = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=15)
                
                if response.status_code != 200:
                    print(f"!! Stopped at {letter} P{current_page} (Status: {response.status_code})")
                    return

                soup = BeautifulSoup(response.text, 'html.parser')
                tiles = soup.find_all('div', class_='product-tile')

                if not tiles:
                    print(f"Finished all pages for letter '{letter}'.")
                    break

                # Extract and save links immediately
                page_links = []
                for tile in tiles:
                    link_tag = tile.find('a', href=True)
                    if link_tag:
                        page_links.append(link_tag['href'])
                
                append_links(page_links)
                
                # Update progress state
                current_page += 1
                save_progress(l_idx, current_page)
                
                # Anti-throttle delay
                time.sleep(3.5)

            except Exception as e:
                print(f"An error occurred: {e}. Progress saved. Try running again later.")
                return

        # Reset start_page for the next letter in the alphabet loop
        start_page = 1

    print("Scraping Complete! Progress file can be deleted.")

if __name__ == "__main__":
    scrape_ewg_with_resume()

Resuming from Letter: e, Page: 23
Scraping Letter 'e', Page 23...
Scraping Letter 'e', Page 24...
Scraping Letter 'e', Page 25...
Scraping Letter 'e', Page 26...
Scraping Letter 'e', Page 27...
Scraping Letter 'e', Page 28...
Scraping Letter 'e', Page 29...
Scraping Letter 'e', Page 30...
Scraping Letter 'e', Page 31...
Scraping Letter 'e', Page 32...
Scraping Letter 'e', Page 33...
Scraping Letter 'e', Page 34...
Scraping Letter 'e', Page 35...
Scraping Letter 'e', Page 36...
Scraping Letter 'e', Page 37...
Scraping Letter 'e', Page 38...
Scraping Letter 'e', Page 39...
Scraping Letter 'e', Page 40...
Scraping Letter 'e', Page 41...
Scraping Letter 'e', Page 42...
Scraping Letter 'e', Page 43...
Scraping Letter 'e', Page 44...
Scraping Letter 'e', Page 45...
Scraping Letter 'e', Page 46...
Scraping Letter 'e', Page 47...
Scraping Letter 'e', Page 48...
Scraping Letter 'e', Page 49...
Scraping Letter 'e', Page 50...
Scraping Letter 'e', Page 51...
Scraping Letter 'e', Page 52...
Scrapi

In [5]:
def deduplicate_links(input_file, output_file):
    print(f"Reading {input_file}...")
    
    try:
        with open(input_file, 'r') as f:
            # Using a set automatically handles uniqueness
            # .strip() removes newlines and extra whitespace
            unique_links = {line.strip() for line in f if line.strip()}
        
        print(f"Found {len(unique_links)} unique product links.")
        
        # Sort them alphabetically for better organization
        sorted_links = sorted(list(unique_links))
        
        with open(output_file, 'w') as f:
            for link in sorted_links:
                f.write(f"{link}\n")
                
        print(f"Cleaned links saved to: {output_file}")
        
    except FileNotFoundError:
        print(f"Error: The file {input_file} was not found.")

if __name__ == "__main__":
    deduplicate_links("ewg_product_links.txt", "unique_ewg_products.txt")

Reading ewg_product_links.txt...
Found 8575 unique product links.
Cleaned links saved to: unique_ewg_products.txt
